# Pipeline mestre — Censo 2022 e análise territorial da RMR

Notebook orquestrador. A lógica analítica deve permanecer em módulos versionados em `src/censo_rmr/`; este notebook controla ambiente, configuração, execução, QA e registro de proveniência.

## 0. Montagem do Google Drive e instalação do projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b agent/pipeline-rmr-v0 https://github.com/hilaliskandar/censo_senso_rmr.git /content/censo_senso_rmr
%cd /content/censo_senso_rmr
!pip -q install -e .


## 1. Configuração central

In [ ]:
from pathlib import Path
from censo_rmr.configuracao import carregar_configuracao

CFG = carregar_configuracao('/content/censo_senso_rmr/config/config.yaml')
DRIVE_RAIZ = Path('/content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR')
assert DRIVE_RAIZ.exists(), f'Pasta do Drive não encontrada: {DRIVE_RAIZ}'
CFG.municipios


## 2. Pré-voo

Esta etapa deverá validar estrutura de pastas, fontes, dicionários e versões antes de qualquer cálculo.

In [ ]:
pastas = CFG.dados['drive']['pastas']
for nome, relativo in pastas.items():
    caminho = DRIVE_RAIZ / relativo
    print(f'{nome:28s}', 'OK' if caminho.exists() else 'AUSENTE', caminho)


## 3. Etapas do pipeline

As chamadas abaixo serão ativadas progressivamente após cada módulo passar pela auditoria e pelos testes de regressão.

1. aquisição/cache IBGE;
2. inspeção de dicionários e esquema;
3. recorte territorial RMR;
4. demografia;
5. composição doméstica;
6. rendimento;
7. FCU e equidade;
8. entorno e condições habitacionais;
9. vulnerabilidade e densidade;
10. Moran/LISA e testes de sensibilidade;
11. segregação;
12. PCA e tipologias;
13. tabelas e mapas;
14. manifesto e relatório de QA.

In [ ]:
ETAPAS_ATIVAS = [
    'pre_voo',
    # 'demografia',
    # 'composicao_domestica',
    # 'renda',
    # demais etapas serão habilitadas após refatoração/auditoria
]
print('Etapas habilitadas:', ETAPAS_ATIVAS)


## 4. Manifesto de execução

In [ ]:
from censo_rmr.proveniencia import manifesto_execucao, salvar_manifesto

manifesto = manifesto_execucao(
    configuracao=CFG.dados,
    avisos=['Pipeline v0: apenas pré-voo habilitado; módulos analíticos em auditoria.'],
)
saida = DRIVE_RAIZ / '08_Relatorio_Consolidado' / 'manifestos' / 'MANIFESTO_EXECUCAO_V0.json'
salvar_manifesto(manifesto, saida)
print(saida)
